In [12]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

In [13]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [14]:
models = {
    "log_reg": LogisticRegression(max_iter=1000),
    "knn": KNeighborsClassifier(n_neighbors=5),
    "tree": DecisionTreeClassifier(random_state=42),
    "rf": RandomForestClassifier(n_estimators=200, random_state=42),
}

In [15]:
rows = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    rows.append({"model": name, "accuracy": acc, "f1": f1})

results = pd.DataFrame(rows).sort_values("f1", ascending=False)
results

,model,accuracy,f1
1,knn,0.973684,0.974321
0,log_reg,0.947368,0.948718
2,tree,0.894737,0.896825
3,rf,0.894737,0.896825


#### Choosing a Default Route

This interpretation is where you decide which route you will recommend to a friend:
“Take the direct train; it is slightly slower than the bus but much more comfortable and predictable.”
The same style of reasoning applies to baseline models.

On this Iris split, **k-Nearest Neighbors (knn)** is the best baseline. It leads with accuracy ≈ 0.974 and F1 ≈ 0.974. Logistic Regression is second (accuracy ≈ 0.947, F1 ≈ 0.949). The gap is about **0.026** on both metrics — a clear but modest edge for knn on this particular train/test cut.

Is the more complex model worth the added complexity? **No.** Random Forest ties the Decision Tree at the bottom (≈ 0.895) and loses to both knn and Logistic Regression. Extra trees and ensembles do not buy better results here, so the “fancier bus” is not the route to recommend.

The calm default I would suggest to a friend: **start with knn** as the working baseline on this split. If simplicity and a stable story matter more than squeezing the last points, Logistic Regression is a close second and still beats the tree-based options.

If this were a real project, next I would:
1. check the ranking with cross-validation so one lucky split cannot decide alone,
2. try a few values of `k` for knn (and keep Logistic Regression as the simple backup),
3. only then revisit Random Forest if a clear, stable gap still appears after those checks.